In [ ]:
# Cell 1: Import required libraries and configure Google Gemini API
import pandas as pd
import numpy as np
import google.generativeai as genai
import json
import os

# Define the Gemini API Key 
# Authentication
GOOGLE_API_KEY = "AQ.Ab8RN6INtWpGP-qJObHsycJgbOkPNWw00_8QOezNM5SaytIXqQ"
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the fast, lightweight and free model (gemini-1.5-flash)
model = genai.GenerativeModel('gemini-1.5-flash')

In [ ]:
# Define relative paths and load raw datasets into Pandas DataFrames
rekrute_path = os.path.join('..', 'data', 'raw', 'Rekrute.csv')
emploi_path = os.path.join('..', 'data', 'raw', 'emploi_ma_jobs.csv')

# Load the CSV files into memory
df_rekrute = pd.read_csv(rekrute_path, sep=',')
df_emploi = pd.read_csv(emploi_path, sep=';')

print(f" ReKrute raw row count : {len(df_rekrute)}")
print(f" Emploi.ma raw row count : {len(df_emploi)}")

 ReKrute raw row count : 10782
 Emploi.ma raw row count : 63


In [7]:
# Cell 3: Advanced extraction function for 8 strategic fields
def extract_job_details(description_text):
    if pd.isna(description_text) or str(description_text).strip() == "":
        return {
            "salaire_min": None, "salaire_max": None, "contrat": None, "teletravail": "On-site",
            "programming_languages": [], "bi_tools": [], "minimum_degree": "Not Specified", "english_level": "Not Specified"
        }
    
    prompt = f"""
    Analyse this Moroccan tech job description and extract STRICTLY a valid JSON object with these exact keys:
    - "salaire_min": (int or null, monthly net salary in MAD)
    - "salaire_max": (int or null, monthly net salary in MAD)
    - "contrat": (only choose from: "CDI", "CDD", "Stage", "Freelance", or null)
    - "teletravail": (only choose from: "Remote", "Hybrid", "On-site")
    - "programming_languages": (Array of strings, e.g., ["Python", "SQL"], extract from text)
    - "bi_tools": (Array of strings, e.g., ["Power BI", "Tableau"], extract from text)
    - "minimum_degree": (only choose from: "Bac+2", "Bac+3", "Bac+5", "Not Specified")
    - "english_level": (only choose from: "Courant", "Technique", "Not Specified")

    Text to analyse:
    {description_text}

    Return ONLY the JSON object. Do not include markdown formatting or backticks.
    """
    
    try:
        response = model.generate_content(prompt)
        clean_text = response.text.replace('```json', '').replace('```', '').strip()
        return json.loads(clean_text)
    except:
        return {"salaire_min": None, "salaire_max": None, "contrat": None, "teletravail": "On-site", "programming_languages": [], "bi_tools": [], "minimum_degree": "Not Specified", "english_level": "Not Specified"}

print("✅ Cell 3: Function ready!")

✅ Cell 3: Function ready!


In [8]:
# Cell 4: Test the intelligent extraction function on a sample row from ReKrute

# 1. Grab the job description text from the very first row (index 0)
sample_text = df_rekrute['description_poste'].iloc[0]

# 2. Print the original raw text just to see what we are analyzing
print("📝 [ORIGINAL RAW TEXT FROM REKRUTE]:")
print(sample_text[:500] + "...\n") # Print only first 500 characters to keep it clean

print("="*50 + "\n")

# 3. Run our smart Gemini function on this text
extracted_sample_result = extract_job_details(sample_text)

# 4. Display the clean structured JSON output
import pprint
print("🎯 [GEMINI AI EXTRACTED STRUCTURED RESULT]:")
pprint.pprint(extracted_sample_result)

📝 [ORIGINAL RAW TEXT FROM REKRUTE]:
Une nouvelle activité a pris naissance au sein de notre centre de service AtoS Casablanca, il s’agit de la cellule PMO. Cette cellule offre des services de support, gestion et de Reporting au TOP Management Atos France. Cette activité Permet aux PMOs de la cellule de maitriser les processus de Management au sein d’ATOS et ouvre donc les portes à ces personnes vers des postes à haute responsabilité. La mission consiste à Industrialiser certains de nos Processus, ainsi fournir un certain volume de...


🎯 [GEMINI AI EXTRACTED STRUCTURED RESULT]:
{'bi_tools': [],
 'contrat': None,
 'english_level': 'Not Specified',
 'minimum_degree': 'Not Specified',
 'programming_languages': [],
 'salaire_max': None,
 'salaire_min': None,
 'teletravail': 'On-site'}
